<a href="https://colab.research.google.com/github/ghiorsobeyene367-source/Classificator/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9C%D0%B0%D0%B3%D0%B8%D1%81%D1%82%D0%B5%D1%80%D1%81%D0%BA%D0%B0%D1%8F_%D0%9A%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%82%D0%BE%D1%80%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1: УСТАНОВКА ЗАВИСИМОСТЕЙ

!pip install -q sentence-transformers umap-learn

#2: ИМПОРТ БИБЛИОТЕК И НАСТРОЙКА ОКРУЖЕНИЯ

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore') # Отключаем лишние предупреждения для чистоты вывода

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


import transformers
from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import umap

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import folium


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Автоопределение устройства (GPU или CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Вычисления будут производиться на: {device}")

In [ ]:
#3 (ШАГ 1): ЗАГРУЗКА ДАННЫХ
import os
import pandas as pd

# пути к файлам в Google Colab
from google.colab import drive
drive.mount('/content/drive')
HACKER_FILE = '/content/TheHackerNews_Dataset (1).xlsx'
REPORTS_FILE = '/content/reports.csv'

# Проверка наличия файлов
if not os.path.exists(HACKER_FILE) or not os.path.exists(REPORTS_FILE):
    print(f"⚠️ ВНИМАНИЕ: Файлы датасетов не найдены по указанным путям!")
    try:
        from google.colab import files
        print("Открываю диалоговое окно для загрузки. Выберите оба файла...")
        uploaded = files.upload()
    except ImportError:
        pass

print("Чтение сырых файлов (происходит моментально)...")

# 1. Чтение HackerNews (Excel формат)
try:
    df_hn_raw = pd.read_excel(HACKER_FILE)
    print(f"✅ HackerNews загружен: {df_hn_raw.shape[0]} строк.")
except Exception as e:
    print(f"❌ Ошибка HackerNews: {e}")
    try:
        df_hn_raw = pd.read_csv(HACKER_FILE, on_bad_lines='skip', encoding='utf-8')
        print(f"✅ HackerNews загружен (через CSV-фоллбек): {df_hn_raw.shape[0]} строк.")
    except Exception as e2:
        df_hn_raw = pd.DataFrame()

# 2. Чтение Reports
try:
    df_rep_raw = pd.read_csv(REPORTS_FILE, encoding='utf-8')
    print(f"✅ Reports загружен: {df_rep_raw.shape[0]} строк.")
except UnicodeDecodeError:
    df_rep_raw = pd.read_csv(REPORTS_FILE, encoding='latin-1')
    print(f"✅ Reports загружен (с кодировкой latin-1): {df_rep_raw.shape[0]} строк.")
except Exception as e:
    print(f"❌ Ошибка Reports: {e}")
    df_rep_raw = pd.DataFrame()

In [ ]:

#3 (ШАГ 2): ПРЕДОБРАБОТКА И ОЧИСТКА ДАННЫХ

def clean_text(text):
    """Глубокая очистка текстовых данных от мусора."""
    if not isinstance(text, str): return ""
    text = re.sub(r'<[^>]+>', ' ', text) # Удаляем HTML теги
    text = re.sub(r'http[s]?://\S+', ' ', text) # Удаляем URL ссылки
    text = re.sub(r'[^a-zA-Zа-яА-Я0-9\s.,!?]', '', text) # Оставляем только буквы, цифры и базовую пунктуацию
    text = re.sub(r'\s+', ' ', text).strip() # Убираем двойные пробелы
    return text

print("Начало предобработки и очистки текстов (может занять некоторое время)...")

# Обработка HackerNews
if not df_hn_raw.empty:
    if 'Title' not in df_hn_raw: df_hn_raw['Title'] = ''
    if 'Article' not in df_hn_raw: df_hn_raw['Article'] = ''
    df_hn_raw['Title'] = df_hn_raw['Title'].fillna('')
    df_hn_raw['Article'] = df_hn_raw['Article'].fillna('')
    df_hn = pd.DataFrame({
        'text': df_hn_raw['Title'].astype(str) + ". " + df_hn_raw['Article'].astype(str),
        'source': 'HackerNews'
    })
else:
    df_hn = pd.DataFrame(columns=['text', 'source'])

# Обработка Reports
if not df_rep_raw.empty:
    if 'title' not in df_rep_raw: df_rep_raw['title'] = ''
    if 'text' not in df_rep_raw: df_rep_raw['text'] = ''
    df_rep_raw['title'] = df_rep_raw['title'].fillna('')
    df_rep_raw['text'] = df_rep_raw['text'].fillna('')
    df_rep = pd.DataFrame({
        'text': df_rep_raw['title'].astype(str) + ". " + df_rep_raw['text'].astype(str),
        'source': 'AI_Reports'
    })
else:
    df_rep = pd.DataFrame(columns=['text', 'source'])

# Объединение
df_master = pd.concat([df_hn, df_rep], ignore_index=True)

if df_master.empty:
    raise ValueError("Датасет пуст! Убедитесь, что файлы успешно загружены в Шаге 1.")

# Очистка текста
df_master['text'] = df_master['text'].apply(clean_text)

# Фильтрация артефактов (короткие тексты и дубликаты)
df_master = df_master[df_master['text'].str.len() > 100]
df_master.drop_duplicates(subset=['text'], inplace=True)
df_master.reset_index(drop=True, inplace=True)

print(f"✅ Данные успешно подготовлены и объединены! Итоговый размер: {df_master.shape[0]} записей.")

In [ ]:
# 4: ИЗВЛЕЧЕНИЕ СЕМАНТИЧЕСКИХ ПРИЗНАКОВ (LLM EMBEDDINGS)

MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
print(f"Загрузка языковой модели {MODEL_NAME} через HuggingFace...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
embed_model = AutoModel.from_pretrained(MODEL_NAME).to(device)

def get_embeddings(texts, batch_size=32):
    """Генерация эмбеддингов батчами (чтобы не переполнить память GPU)."""
    embed_model.eval()
    all_embeddings = []

    print("Генерация векторных представлений (эмбеддингов)... Это может занять пару минут.")
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]

            # Токенизация
            encoded_input = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt').to(device)

            # Получение выхода модели
            model_output = embed_model(**encoded_input)

            # Mean Pooling (усреднение векторов токенов для получения вектора предложения)
            attention_mask = encoded_input['attention_mask']
            token_embeddings = model_output[0]
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

            batch_embeddings = sum_embeddings / sum_mask
            all_embeddings.append(batch_embeddings.cpu().numpy())

            if (i // batch_size + 1) % 10 == 0:
                print(f"Обработано батчей: {i // batch_size + 1} / {len(texts) // batch_size + 1}")

    return np.vstack(all_embeddings)

embeddings = get_embeddings(df_master['text'].tolist())
print(f"✅ Размерность полученной матрицы признаков: {embeddings.shape}")

In [ ]:
# 5: КЛАСТЕРИЗАЦИЯ И ИНТЕРПРЕТАЦИЯ (РАЗМЕТКА ДЛЯ ДИССЕРТАЦИИ)


NUM_CLUSTERS = 4
print(f"Выполнение алгоритма K-Means (Поиск {NUM_CLUSTERS} кластеров)...")
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=SEED, n_init=10)
df_master['cluster'] = kmeans.fit_predict(embeddings)

# Благодаря фиксации SEED=42, номера кластеров стабильны.
CLUSTER_NAMES = {
    0: "Автономные системы и физ. риски",
    1: "Уязвимости генеративного ИИ (LLM)",
    2: "Классические кибератаки и ВПО",
    3: "Deepfake и медиафальсификации"
}
df_master['cluster_name'] = df_master['cluster'].map(CLUSTER_NAMES)

def extract_top_keywords(df, num_words=10):
    """Извлекает ключевые слова для каждого кластера с помощью TF-IDF."""
    print("\n--- ТОП КЛЮЧЕВЫХ СЛОВ ПО КЛАСТЕРАМ ---")

    # Расширяем стандартный список стоп-слов нашими "словами-паразитами"
    custom_stop_words = list(ENGLISH_STOP_WORDS) + [
        'said', 'people', 'company', 'use', 'also', 'like', 'just', 'new', 'time', 'told', 'according', 'says'
    ]

    # Используем обновленный список стоп-слов
    tfidf = TfidfVectorizer(stop_words=custom_stop_words, max_features=5000)

    for cluster_id in range(NUM_CLUSTERS):
        cluster_texts = df[df['cluster'] == cluster_id]['text']
        tfidf_matrix = tfidf.fit_transform(cluster_texts)

        # Получаем среднее значение tf-idf для каждого слова в кластере
        mean_tfidf = tfidf_matrix.mean(axis=0).A1
        words = tfidf.get_feature_names_out()

        top_idx = mean_tfidf.argsort()[-num_words:][::-1]
        top_words = [words[i] for i in top_idx]

        # Печатаем с красивым названием
        cluster_label = CLUSTER_NAMES.get(cluster_id, f"Кластер {cluster_id}")
        print(f"[{cluster_label}]")
        print(f"Ключевые слова: {', '.join(top_words)}")
        print(f"Количество записей: {len(cluster_texts)}\n")

extract_top_keywords(df_master)

In [ ]:
# 6: ВИЗУАЛИЗАЦИЯ ПРОСТРАНСТВА УГРОЗ (UMAP)

print("Снижение размерности для визуализации (UMAP)...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=SEED)
umap_emb = reducer.fit_transform(embeddings)

plt.figure(figsize=(14, 9))
scatter = sns.scatterplot(x=umap_emb[:, 0], y=umap_emb[:, 1],
                          hue=df_master['cluster_name'], palette='tab10', s=40, alpha=0.8, edgecolor='k')

plt.title('Семантическое пространство угроз ИИ и Кибербезопасности (UMAP)', fontsize=16, fontweight='bold')
plt.xlabel('UMAP Компонента 1', fontsize=12)
plt.ylabel('UMAP Компонента 2', fontsize=12)

# Настройка легенды
plt.legend(title='Категория Угрозы', title_fontsize='13', fontsize='11', loc='best')
plt.grid(True, linestyle='--', alpha=0.5)

plt.savefig('umap_clusters_high_res.png', dpi=300, bbox_inches='tight')
print("✅ График кластеров сохранен как 'umap_clusters_high_res.png'")
plt.show()

In [ ]:
# 7: АРХИТЕКТУРА НЕЙРОСЕТИ НА PYTORCH

class AIThreatNet(nn.Module):
    """Полносвязная глубокая нейронная сеть для классификации текстов."""
    def __init__(self, input_size, num_classes):
        super(AIThreatNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4) # Dropout для предотвращения переобучения

        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.drop2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(256, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.drop1(self.relu(self.bn1(self.fc1(x))))
        x = self.drop2(self.relu(self.bn2(self.fc2(x))))
        x = self.fc3(x)
        return x

In [ ]:
# 8: ПОДГОТОВКА И ОБУЧЕНИЕ МОДЕЛИ

# 1. Разделение выборки
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, df_master['cluster'].values, test_size=0.2, random_state=SEED, stratify=df_master['cluster'].values
)

# 2. Перевод в тензоры PyTorch
train_loader = DataLoader(TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).long()),
                          batch_size=64, shuffle=True)
test_tensor_X = torch.tensor(X_test).float().to(device)
test_tensor_y = torch.tensor(y_test).long().to(device)

# 3. Инициализация модели, функции потерь и оптимизатора
input_dim = embeddings.shape[1]
model = AIThreatNet(input_dim, NUM_CLUSTERS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 4. Цикл обучения (Training Loop)
EPOCHS = 25
train_losses = []

print("Начало обучения нейронной сети...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Эпоха [{epoch+1}/{EPOCHS}] | Функция потерь (Loss): {avg_loss:.4f}")

In [ ]:
# 9: ОЦЕНКА РЕЗУЛЬТАТОВ И ГРАФИКИ ДЛЯ ДИССЕРТАЦИИ

# 1. График обучения (Loss curve)
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, marker='o', color='b', label='Training Loss')
plt.title('Кривая обучения нейросети (Training Loss)', fontsize=14)
plt.xlabel('Эпоха')
plt.ylabel('Значение Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig('training_loss_high_res.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Оценка на тестовой выборке
model.eval()
with torch.no_grad():
    test_preds = model(test_tensor_X)
    _, predicted_classes = torch.max(test_preds, 1)

y_pred = predicted_classes.cpu().numpy()

# Получаем список имен для отчета и матрицы
target_names = [CLUSTER_NAMES[i] for i in range(NUM_CLUSTERS)]

print("\n=== ОТЧЕТ О КЛАССИФИКАЦИИ (CLASSIFICATION REPORT) ===")
print(classification_report(y_test, y_pred, target_names=target_names))

# 3. Матрица ошибок (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
# Добавлено использование имен вместо цифр 0,1,2,3
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names,
            yticklabels=target_names)

plt.title('Матрица ошибок (Confusion Matrix)', fontsize=14)
plt.ylabel('Истинный класс', fontsize=12)
plt.xlabel('Предсказанный класс', fontsize=12)
# Поворачиваем метки оси X, чтобы длинные названия не наезжали друг на друга
plt.xticks(rotation=45, ha='right')

plt.savefig('confusion_matrix_high_res.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Анализ завершен! Все графики сохранены в рабочей области Colab.")

In [ ]:
# 10: ГЕОГРАФИЧЕСКИЙ АНАЛИЗ (РОССИЯ, США, КИТАЙ)

print("\nИзвлечение гео-данных (Россия, США, Китай) и временных меток из текстов...")

country_data = []
for idx, row in df_master.iterrows():
    text_lower = row['text'].lower()

    # Регулярное выражение для извлечения года из текста (от 2010 до 2025)
    years = re.findall(r'\b(201[0-9]|202[0-5])\b', text_lower)
    year = int(years[0]) if years else None

    # Ищем упоминания стран
    if any(w in text_lower for w in ['russia', 'russian', 'moscow', 'kremlin']):
        country_data.append({'Country': 'Россия', 'Cluster': row['cluster_name'], 'Year': year})
    if any(w in text_lower for w in ['usa', 'us', 'united states', 'america', 'american', 'washington']):
        country_data.append({'Country': 'США', 'Cluster': row['cluster_name'], 'Year': year})
    if any(w in text_lower for w in ['china', 'chinese', 'beijing']):
        country_data.append({'Country': 'Китай', 'Cluster': row['cluster_name'], 'Year': year})

df_geo = pd.DataFrame(country_data)
print(f"Найдено упоминаний стран: Россия - {len(df_geo[df_geo['Country']=='Россия'])}, США - {len(df_geo[df_geo['Country']=='США'])}, Китай - {len(df_geo[df_geo['Country']=='Китай'])}")

# 1. Сравнение угроз по странам (Bar Chart)
plt.figure(figsize=(12, 7))
sns.countplot(data=df_geo, x='Country', hue='Cluster', palette='Set2')
plt.title('Распределение типов угроз ИИ по странам (Россия, США, Китай)', fontsize=16)
plt.xlabel('Страна', fontsize=14)
plt.ylabel('Количество упоминаний', fontsize=14)
plt.legend(title='Тип угрозы', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('geo_threats_bar.png', dpi=300)
plt.show()

# 2. Heatmap угроз по странам
plt.figure(figsize=(10, 6))
geo_pivot = pd.crosstab(df_geo['Cluster'], df_geo['Country'])
sns.heatmap(geo_pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=.5)
plt.title('Тепловая карта интенсивности угроз (Heatmap)', fontsize=16)
plt.xlabel('Страна', fontsize=12)
plt.ylabel('Тип угрозы', fontsize=12)
plt.tight_layout()
plt.savefig('geo_heatmap.png', dpi=300)
plt.show()

# 3. Динамика интенсивности упоминаний по годам
df_geo_years = df_geo.dropna(subset=['Year'])


df_geo_years = df_geo_years[df_geo_years['Year'] <= 2024]

if not df_geo_years.empty:
    plt.figure(figsize=(12, 6))
    sns.histplot(data=df_geo_years, x='Year', hue='Country', multiple="stack", bins=len(df_geo_years['Year'].unique()), palette='Set1')
    plt.title('Динамика упоминаний угроз с участием ИИ (2010 - 2024)', fontsize=16)
    plt.xlabel('Год', fontsize=14)
    plt.ylabel('Количество инцидентов / упоминаний', fontsize=14)
    plt.xticks(range(int(df_geo_years['Year'].min()), int(df_geo_years['Year'].max())+1), rotation=45)
    plt.tight_layout()
    plt.savefig('geo_dynamics.png', dpi=300)
    plt.show()

In [ ]:
# 11: ДОПОЛНИТЕЛЬНЫЕ ВИЗУАЛИЗАЦИИ (WORDCLOUD И КАРТА)



CUSTOM_STOP_WORDS = list(ENGLISH_STOP_WORDS) + [
    'said', 'people', 'company', 'use', 'also', 'like', 'just', 'new', 'time', 'told', 'according', 'says', 'did', 'make'
]

print("\nГенерация облака слов (WordCloud) для каждого кластера...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i in range(NUM_CLUSTERS):
    cluster_texts = " ".join(df_master[df_master['cluster'] == i]['text'].tolist())
    wordcloud = WordCloud(width=800, height=400, background_color='white',
                          stopwords=CUSTOM_STOP_WORDS, max_words=50, colormap='viridis').generate(cluster_texts)

    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].set_title(CLUSTER_NAMES[i], fontsize=14, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('wordclouds_clusters.png', dpi=300)
plt.show()

print("\nСоздание интерактивной карты (Folium)...")
# Базовые координаты стран
geo_coords = {
    'Россия': [61.52, 105.31],
    'США': [37.09, -95.71],
    'Китай': [35.86, 104.19]
}

# Создаем базовую карту
m = folium.Map(location=[40, 0], zoom_start=2, tiles='CartoDB positron')

for country in geo_coords.keys():
    # Находим топ-3 угрозы для страны
    top_threats = df_geo[df_geo['Country'] == country]['Cluster'].value_counts().head(3)

    # Формируем HTML для всплывающего окна
    popup_html = f"<b>{country}</b><br><hr><b>ТОП-3 Угрозы ИИ:</b><br>"
    for threat, count in top_threats.items():
        popup_html += f"- {threat}: <i>{count} шт.</i><br>"

    folium.Marker(
        location=geo_coords[country],
        popup=folium.Popup(popup_html, max_width=300),
        icon=folium.Icon(color='red', icon='info-sign'),
        tooltip=f"Нажмите для анализа угроз в регионе: {country}"
    ).add_to(m)

# Сохраняем карту
m.save("threats_map.html")
print("✅ Интерактивная карта успешно создана! Она сохранена как файл 'threats_map.html'.")

In [ ]:

# 13: ИНТЕРАКТИВНОЕ ПРЕДСКАЗАНИЕ ДЛЯ ПОЛЬЗОВАТЕЛЬСКОГО ТЕКСТА

import ipywidgets as widgets
from IPython.display import display, HTML
import torch.nn.functional as F # Добавлено для полной независимости ячейки

print("\nИнициализация интерактивного модуля предсказания...")

# Создаем графические элементы интерфейса для Colab
text_area = widgets.Textarea(
    value='',
    placeholder='Вставьте текст новости или отчета об инциденте на английском языке...',
    description='Текст:',
    layout=widgets.Layout(width='90%', height='150px')
)

predict_button = widgets.Button(
    description='Классифицировать',
    button_style='success',
    tooltip='Нажмите для анализа текста нейросетью',
    icon='search'
)

output_area = widgets.Output()

def on_predict_button_clicked(b):
    with output_area:
        output_area.clear_output()
        user_text = text_area.value

        if len(user_text.split()) < 5:
            print("⚠️ Пожалуйста, введите более длинный осмысленный текст (минимум 5-10 слов).")
            return

        print("⏳ Нейросеть анализирует текст...")

        # 1. Очистка текста
        cleaned = clean_text(user_text)

        # 2. Векторизация (Эмбеддинг через HuggingFace)
        embed_model.eval()
        with torch.no_grad():
            encoded = tokenizer([cleaned], padding=True, truncation=True, return_tensors='pt').to(device)
            model_out = embed_model(**encoded)
            mask = encoded['attention_mask'].unsqueeze(-1).expand(model_out[0].size()).float()
            sum_emb = torch.sum(model_out[0] * mask, 1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            embedding = sum_emb / sum_mask

        # 3. Предсказание (Классификация обученной нейросетью)
        model.eval()
        with torch.no_grad():
            logits = model(embedding)
            probs = F.softmax(logits, dim=1)[0]
            pred_class = torch.argmax(logits, 1).item()
            confidence = probs[pred_class].item() * 100

        # 4. Вывод красивого результата
        print("\n" + "=" * 60)
        print(f"🎯 РЕЗУЛЬТАТ НЕЙРОСЕТИ:")
        print(f"Категория угрозы:  {CLUSTER_NAMES[pred_class].upper()}")
        print(f"Уверенность:       {confidence:.2f}%")
        print("=" * 60)

        # Дополнительно выводим вероятности по всем классам
        print("\nРаспределение вероятностей по всем кластерам:")
        for i, p in enumerate(probs):
            print(f"- {CLUSTER_NAMES[i]}: {p.item()*100:.1f}%")

predict_button.on_click(on_predict_button_clicked)

# Отображение интерфейса
display(HTML("<h3 style='color: #2c3e50;'>Анализатор новых угроз (Интерактивный тест)</h3>"))
display(text_area)
display(predict_button)
display(output_area)

In [ ]:
# 14: ЭКСПОРТ МОДЕЛИ И ДАННЫХ ДЛЯ WEB-ПРИЛОЖЕНИЯ (STREAMLIT MVP)

import joblib

print("\nПодготовка к экспорту модели для создания Web-приложения...")

# 1. Сохраняем веса обученной нейросети
MODEL_SAVE_PATH = 'ai_threat_model.pth'
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Веса модели сохранены в {MODEL_SAVE_PATH}")

# 2. Сохраняем объект кластеризатора (K-Means), чтобы приложение знало центры кластеров
KMEANS_SAVE_PATH = 'kmeans_clusterer.pkl'
joblib.dump(kmeans, KMEANS_SAVE_PATH)
print(f"Кластеризатор K-Means сохранен в {KMEANS_SAVE_PATH}")

# 3. Сохраняем базовый датасет гео-угроз, чтобы приложение могло отрисовать первую карту
GEO_DATA_PATH = 'geo_threats_data.csv'
df_geo.to_csv(GEO_DATA_PATH, index=False)
print(f"Гео-данные сохранены в {GEO_DATA_PATH}")

# Автоматическое скачивание файлов экспорта
try:
    from google.colab import files
    files.download(MODEL_SAVE_PATH)
    files.download(KMEANS_SAVE_PATH)
    files.download(GEO_DATA_PATH)
except:
    pass